# CN7 가스: 같은 Run 정상 관계 공분산 보조 모델
개별 범위 안에서도 센서 조합이 드문지 검토한다. 정상 기반 보조 모델임을 구분한다.


In [1]:
import nbformat
old=nbformat.read("notebooks/170_cn7_gas_persistent_relation_experts.ipynb",as_version=4)
oldcode="\n".join(c.source for c in old.cells if c.cell_type=="code")
exec(oldcode[:oldcode.index("def predict(parent,q):")])
exec(oldcode[oldcode.index("def stats(a,b):"):oldcode.index("records=[];choices=[]")])
from sklearn.covariance import LedoitWolf
V=["Mold_Temperature_3","Mold_Temperature_4","Max_Injection_Speed","Filling_Time",
 "Max_Injection_Pressure","Max_Switch_Over_Pressure","Plasticizing_Position","Plasticizing_Time","Cushion_Position"]
raw=d[[v+"_raw" for v in V]].to_numpy(float)
def views(ctx):
 ii=np.flatnonzero(ctx);a=pd.DataFrame(raw[ii],index=ii)
 segment=np.cumsum(np.r_[True,(g[ii][1:]!=g[ii][:-1])|(np.diff(ii)!=1)])
 past=a.groupby(segment,group_keys=False).apply(lambda b:b.shift(1).rolling(5,min_periods=3).median())
 delta=a-past
 return {"flow6":a.iloc[:,:6],"dynamic_flow6":delta.iloc[:,:6],
 "meter3":a.iloc[:,6:],"thermal_speed3":a.iloc[:,[0,1,2]]}
memo={};audit=[]
def predict(parent,q):
 key=(tuple(np.flatnonzero(parent)),tuple(q))
 if key in memo:return memo[key]
 ctx=train_context(parent,q);ref=np.flatnonzero(ctx&eligible)
 assert not ctx[q].any() and set(ref).isdisjoint(q)
 audit.append({"timestamp_overlap":len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q])),
 "test_raw_in_training_history":int(ctx[q].sum())})
 tr=views(ctx);te=views(parent);out={}
 for view in tr:
  for method in ["LW","ridge01","ridge001"]:
   out[view+"_"+method]=np.zeros(len(q))
  for run in pd.unique(g[q]):
   normal=ref[(y[ref]==0)&(g[ref]==run)]
   if len(normal)<8:normal=ref[y[ref]==0]
   pos=np.flatnonzero(g[q]==run)
   a=tr[view].reindex(normal).to_numpy(float);b=te[view].reindex(q[pos]).to_numpy(float)
   imp=SimpleImputer(keep_empty_features=True).fit(a);a=imp.transform(a);b=imp.transform(b)
   floor=[]
   for k in range(a.shape[1]):
    dif=np.diff(np.unique(np.round(a[:,k],4)));nz=dif[dif>1e-8]
    floor.append(max(float(np.min(nz)) if len(nz) else .01,1e-4))
   med=np.median(a,axis=0);scale=np.maximum(a.std(axis=0),floor)
   a=np.clip((a-med)/scale,-20,20);b=np.clip((b-med)/scale,-20,20)
   out[view+"_LW"][pos]=LedoitWolf().fit(a).mahalanobis(b)
   center=a.mean(axis=0);cov=np.cov(a,rowvar=False)
   for ridge in [.1,.01]:
    inv=np.linalg.pinv(cov+ridge*np.eye(cov.shape[0]))
    err=b-center
    name="ridge01" if ridge==.1 else "ridge001"
    out[view+"_"+name][pos]=np.einsum("ij,jk,ik->i",err,inv,err)
 memo[key]=(out,ref,ctx);return memo[key]
b=pd.read_csv(SOURCE/"cn7_165_gas_supervised_complementary_predictions.csv",parse_dates=["TimeStamp"]).set_index("TimeStamp").loc[d.TimeStamp]
base165=b.three_specialists_confirmed.to_numpy(int)
a=pd.read_csv(OUT/"cn7_170_persistence_predictions.csv",parse_dates=["TimeStamp"])
a=a[(a.model=="persistent_abs4_ET")&(a.rule=="normal95")].set_index("TimeStamp").loc[d.TimeStamp]
base170=base165|a.prediction.to_numpy(int)
a=pd.read_csv(OUT/"cn7_171_flow_predictions.csv",parse_dates=["TimeStamp"])
a=a[(a.model=="flow_signed6_LR")&(a.rule=="normal99")].set_index("TimeStamp").loc[d.TimeStamp]
base171=base170|a.prediction.to_numpy(int)
a=pd.read_csv(OUT/"cn7_173_transfer_predictions.csv",parse_dates=["TimeStamp"])
a=a[(a.model=="mechanism_joint8_LR")&(a.rule=="inner_normal0.99")].set_index("TimeStamp").loc[d.TimeStamp]
base173=base170|a.prediction.to_numpy(int)
records=[];choices=[]
for held in pd.unique(blocks):
 q=np.flatnonzero(blocks==held);outer,ref,ctx=predict(np.ones(N,bool),q)
 inner={n:np.full(len(ref),np.nan) for n in outer}
 for ib in pd.unique(innerblocks[ref]):
  mask=innerblocks[ref]==ib;ip,_,_=predict(ctx,ref[mask])
  for n in inner:inner[n][mask]=ip[n]
 options=[]
 for name in outer:
  for quant in [.97,.99,.995]:
   cut=float(np.nextafter(np.quantile(inner[name][y[ref]==0],quant,method="higher"),np.inf))
   met=stats(y[ref],inner[name]>=cut)
   options.append((met["F1"],met["TP"],-met["FP"],name,quant,cut))
   for j,i in enumerate(q):records.append({"model":name,"rule":f"inner_normal{quant}",
    "TimeStamp":d.TimeStamp.iloc[i],"actual":int(y[i]),"eligible":int(eligible[i]),
    "score":float(outer[name][j]),"threshold":cut,"prediction":int(outer[name][j]>=cut)})
 best=max(options);_,_,_,name,quant,cut=best
 choices.append({"heldout_block":held,"model":name,"quantile":quant,"inner_F1":best[0]})
 for j,i in enumerate(q):records.append({"model":"inner_selected","rule":"inner_selection",
  "TimeStamp":d.TimeStamp.iloc[i],"actual":int(y[i]),"eligible":int(eligible[i]),
  "score":float(outer[name][j]),"threshold":cut,"prediction":int(outer[name][j]>=cut)})
 print(f"Finished covariance {len(choices)}/20",flush=True)
pp=pd.DataFrame(records);rows=[]
for (name,rule),a in pp.groupby(["model","rule"],sort=False):
 a=a.set_index("TimeStamp").loc[d.TimeStamp];pr=a.prediction.to_numpy(int)
 for policy,base in [("standalone",np.zeros(N,int)),("add_to_165",base165),("add_to_170",base170),("add_to_171",base171),("add_to_173",base173)]:
  rows.append({"model":name,"rule":rule,"policy":policy,"N":605,"gas":13,**stats(y[eligible],(pr|base)[eligible])})
mm=pd.DataFrame(rows)
pp.to_csv(OUT/"cn7_174_covariance_predictions.csv",index=False,encoding="utf-8-sig")
mm.to_csv(OUT/"cn7_174_covariance_metrics.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(audit).to_csv(OUT/"cn7_174_covariance_audit.csv",index=False)
pd.DataFrame(choices).to_csv(OUT/"cn7_174_covariance_inner_choices.csv",index=False)
assert pd.DataFrame(audit).timestamp_overlap.eq(0).all()
assert pd.DataFrame(audit).test_raw_in_training_history.eq(0).all()
top=mm.sort_values(["F1","FP"],ascending=[False,True]).head(15)
best=top.iloc[0]
a=pp[(pp.model==best.model)&(pp.rule==best.rule)].set_index("TimeStamp").loc[d.TimeStamp]
base={"standalone":np.zeros(N,int),"add_to_165":base165,"add_to_170":base170,"add_to_171":base171,"add_to_173":base173}[best.policy]
pr=a.prediction.to_numpy(int)|base
cases=d[["TimeStamp","gas","any_fail","run_id","seq"]].copy();cases["baseline"]=base;cases["candidate"]=pr
cases.to_csv(OUT/"cn7_174_covariance_best_cases.csv",index=False,encoding="utf-8-sig")
print("\nTOP\n",top.to_string(index=False))
print("\nINNER SELECTED\n",mm[mm.model.eq("inner_selected")].to_string(index=False))
print("\nGAS\n",cases[cases.gas].to_string(index=False))
lines=["# CN7 가스: 같은 Run 정상 공분산 기반 관계 이상 보조 전문가","",
 "지도학습 기준165/170/171/173에 정상 기반 Mahalanobis 통계 전문가를 추가 비교했다.",
 "현재 충전6/과거5 중앙값 대비 변화6/계량3/금형-속도3 변수군.",
 "LW 수축공분산 및 ridge0.1/0.01 공분산. 모두 학습 정상만 사용한다.",
 "같은20 시간블록/purge10/nested5. 고정 최고는 사후 개발 성적이며 독립 성능이 아니다.","",
 "## 고정 후보",top.to_string(index=False),"",
 "## 안쪽 선택",mm[mm.model.eq("inner_selected")].to_string(index=False),"",
 "## 가스13개",cases[cases.gas].to_string(index=False),"",
 "## 전체606샷",str(stats(y,pr))]
(ROOT/"reports/cn7_174_covariance_review.md").write_text("\n".join(lines),encoding="utf-8")
(OUT/"cn7_174_covariance_provenance.json").write_text(json.dumps({
 "input_sha256":hashlib.sha256(INPUT.read_bytes()).hexdigest(),"validation":"same20/purge10/nested5",
 "cutoffs":"inner normal97/99/99.5; fixed candidates do not use evaluation labels for cutoff fitting",
 "caveat":"test-informed feature development and post-hoc candidate selection",
 "heads":"normal-only covariance; not pure supervised ensemble"},ensure_ascii=False,indent=2),encoding="utf-8")


Finished covariance 1/20


Finished covariance 2/20


Finished covariance 3/20


Finished covariance 4/20


Finished covariance 5/20


...

Finished covariance 18/20


Finished covariance 19/20


Finished covariance 20/20



TOP
                   model              rule     policy   N  gas  TP  FP  FN  TN  precision   recall       F1
          flow6_ridge01 inner_normal0.995 add_to_173 605   13   8   8   5 584   0.500000 0.615385 0.551724
       dynamic_flow6_LW inner_normal0.995 add_to_173 605   13   8   8   5 584   0.500000 0.615385 0.551724
  dynamic_flow6_ridge01 inner_normal0.995 add_to_173 605   13   8   8   5 584   0.500000 0.615385 0.551724
 dynamic_flow6_ridge001 inner_normal0.995 add_to_173 605   13   8   8   5 584   0.500000 0.615385 0.551724
      thermal_speed3_LW inner_normal0.995 add_to_173 605   13   8   8   5 584   0.500000 0.615385 0.551724
 thermal_speed3_ridge01 inner_normal0.995 add_to_173 605   13   8   8   5 584   0.500000 0.615385 0.551724
thermal_speed3_ridge001 inner_normal0.995 add_to_173 605   13   8   8   5 584   0.500000 0.615385 0.551724
          flow6_ridge01 inner_normal0.995 add_to_171 605   13   9  11   4 581   0.450000 0.692308 0.545455
       dynamic_flow6_LW inner_n

384